# tiny-yolov3-hituav on hituav — batch × subset sweep

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lrivals/EmbeddedComputervision/blob/main/notebooks/hituav/tiny-yolov3-hituav_sweep.ipynb)

Role: batch × subset sweep; tier **N** (6 runs of 600 iterations by default); each run is evaluated on `SUBSET` images, then compared with the others. Tasks: [T18.3](../../docs/tasks/M18-jeux-drone.md), [T18.10](../../docs/tasks/M18-jeux-drone.md), [T14.10](../../docs/tasks/M14-notebooks.md);
notebooks: [M14](../../docs/tasks/M14-notebooks.md).

Prerequisites:
- data: `data/hituav/labels/test` — tools/get_datasets.sh hituav (registration: source printed)
- weights: `weights/yolov3-tiny.weights` — tools/get_weights.sh

Each step prints the `python tools/…` command it runs: a result can be reproduced
from the command line. Outputs in `build/notebooks/hituav/tiny-yolov3-hituav/`, never in `results/` (an mAP on `SUBSET`
images is not publishable, M12 rules). Notebook generated by
`python -m tools.notebooks` (`make notebooks`): edit the template
`tools/notebooks/gabarits.py`, not this file.

## Parameters

In [ ]:
NET           = 'build/m11/cfg/tiny-yolov3-hituav-c1.cfg'  # name in CFG_FILES or cfg (relative to the root)
DATASET       = 'hituav'  # key of DATASETS
SPLIT         = None  # None: evaluation split of the dataset
WEIGHTS       = None  # per run: OUT/runs/<run>/final.weights
DEVICE        = 'gpu'  # cpu | gpu (training only; the rest on CPU)
SUBSET        = 50  # images evaluated; 0: full split (tier N)
SIZE          = None  # input WxH (e.g. '640x192'); None: the one of the cfg
RESIZE        = 'stretch'  # stretch (M12 reference) | letterbox
METRIC        = None  # None: the one of the dataset (coco for coco and flir)
DATA_ROOT     = None  # dataset root; None: data/<dataset> (Colab: Drive folder)
BASE          = 'tiny-yolov3-voc'  # base of make_cfg.py
CHANNELS      = 1  # channels of the cfg (1: thermal)
ANCHORS_K     = 6  # anchors of the cfg (6: Tiny-YOLOv3)
INIT          = 'coco'  # coco | he | path of a .weights
INIT_NET      = None  # network of the INIT weights if it differs from NET
BATCHES       = [8, 16, 32]  # batch sizes tried
TRAIN_SUBSETS = [500, 0]  # training images (first n); 0: the whole split
SEEDS         = [0]  # seeds (train.py --seed); T15.2: [1, 2]
ITERS         = 600  # iterations per run (tier N from 600)
SKIP_DONE     = True  # skips the runs that already have final.weights
LR            = 0.001
BURN_IN       = 500
MULTISCALE    = True  # 320-608 every 10 batches
SAVE_EVERY    = None  # None: default of train.py (200)
WORKERS       = 4
DRIVE_DIR     = '/content/drive/MyDrive/EmbeddedCV'  # Colab: archives data/<dataset>.tar, outputs and resume in runs/; None: no Drive
JOBS          = None  # evaluation processes; None: one per CPU core, capped by the memory (Colab TPU runtime: dozens)
SHOW          = 6  # images shown
OUT           = 'build/notebooks/hituav/tiny-yolov3-hituav'  # notebook outputs
REPO_URL      = 'https://github.com/lrivals/EmbeddedComputervision.git'  # Colab: cloned repository
REV           = 'main'  # Colab: revision

## Environment

Local: nothing is installed or downloaded, the cell checks the prerequisites and
stops with the command to run. Colab: clone of the repository, installation, CuPy if
`DEVICE = "gpu"`, weights, then data by `colab.prepare_data` (archive
`<DRIVE_DIR>/data/<dataset>.tar` or direct download; ExDark and FLIR: archive only,
made on the PC by `tools/get_datasets.sh pack <dataset>`).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

COLAB = "google.colab" in sys.modules


def repo_root():
    for d in (Path.cwd(), *Path.cwd().parents):
        if (d / "tools" / "notebooks" / "commandes.py").exists():
            return d
    return None


ROOT = repo_root()
if COLAB:
    if ROOT is None:
        ROOT = Path("/content/EmbeddedComputervision")
        if not ROOT.exists():
            subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    # Clone of an earlier session: reset to REV (notebook and code of the same commit).
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin", REV], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "FETCH_HEAD"], check=True)
    for m in [m for m in sys.modules if m.split(".")[0] in ("tools", "yolo")]:
        del sys.modules[m]  # modules imported before the update
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                    f"{ROOT}/python[data,plots]"], check=True)
    if DEVICE == "gpu":
        # CPU runtime: no driver, CuPy would say "cudaErrorInsufficientDriver".
        try:
            smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        except FileNotFoundError:
            smi = None
        if smi is None or smi.returncode != 0:
            raise RuntimeError("Colab runtime without a GPU: create a new Colab server "
                               "of GPU type (T4, L4 or A100), or DEVICE = 'cpu'")
        cuda = smi.stdout.split("CUDA Version:")[-1].split()[0]  # driver version
        cupy_pkg = "cupy-cuda13x" if int(cuda.split(".")[0]) >= 13 else "cupy-cuda12x"
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", cupy_pkg],
                       check=True)
elif ROOT is None:
    raise RuntimeError("repository root not found: open the notebook from notebooks/")
os.chdir(ROOT)
for p in (ROOT, ROOT / "python"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from tools.notebooks import commandes as C
from tools.notebooks import env

TRACE = env.trace(DEVICE)
env.check_device(DEVICE)
env.prepare(DATASET, INIT if str(INIT).endswith('.weights') else 'weights/yolov3-tiny.weights' if INIT == 'coco' else (), None, DATA_ROOT, COLAB, hint='',
            drive_dir=DRIVE_DIR)
Path(OUT).mkdir(parents=True, exist_ok=True)

## Preparation (datasets other than VOC)

k-means anchors on the training split (`tools/kmeans_anchors.py`, table in `OUT`),
then a cfg with N classes (`tools/make_cfg.py`), as `tools/m11.sh <dataset>-prep`.
Skipped if the cfg already exists.

In [ ]:
import csv
import json

import numpy as np
from IPython.display import Markdown, display
from PIL import Image

from tools.detect import draw
from tools.notebooks.matrice import estimate
from yolo.data import datasets
from yolo.data.letterbox import input_size, parse_size, size_label
from yolo.models.tiny_yolo import load_cfg

if not str(NET).endswith(".cfg"):
    print(f"{NET}: cfg of the repository, no preparation")
elif Path(NET).exists():
    print(f"{NET} exists: preparation skipped")
else:
    anchors = Path(OUT) / f"anchors_{DATASET}{'_' + str(SIZE) if SIZE else ''}.md"
    C.run(C.cmd_anchors(DATASET, SIZE, anchors, DATA_ROOT))
    C.run(C.cmd_make_cfg(BASE, DATASET, C.anchors_k(anchors, ANCHORS_K), NET, SIZE,
                         CHANNELS))

## Preview of the targets

A few augmented images (`yolo.data.loader.VOCDataset`, hence `yolo.data.augment`)
with their boxes: a check of the annotations before hours of computation.

In [ ]:
cfg = load_cfg(NET)
size = input_size(cfg, parse_size(SIZE) if SIZE else None)
channels = cfg["input"][0]
names = datasets.DATASETS[DATASET].classes
samples = datasets.load(DATASET, DATA_ROOT, role="train")
print(f"{len(samples)} training images, input {size_label(size)}, {channels} channel(s)")
from yolo.data.loader import VOCDataset

preview = VOCDataset(samples[:SHOW], train=True, channels=channels)
for k in range(len(preview)):
    chw, boxes, labels = preview.load(k, size, np.random.default_rng(k))
    hwc = (np.clip(chw.transpose(1, 2, 0), 0, 1) * 255).astype(np.uint8)
    img = Image.fromarray(hwc[..., 0] if channels == 1 else hwc)
    display(draw(img, boxes, np.ones(len(labels)), labels, names))

## Sweep plan (T14.10)

One run per cell of the `BATCHES × TRAIN_SUBSETS × SEEDS` grid, every other
parameter equal (those of `tools/m11.sh <dataset>-train`, except `ITERS`).
`TRAIN_SUBSETS` takes the first n images of the training split
(`train.py --subset`), 0 the whole split. A non-zero seed (`train.py --seed`:
heads, batch draw, multiscale) suffixes the run (`b32-sall-g1`); seed 0 keeps the
short name.
At equal iterations, a larger batch sees more images: the "epochs" column shows it.
The learning rate is not scaled with the batch.

In [ ]:
import threading

from tools.notebooks import colab, runs

PLAN = [(b, s, g, runs.run_name(b, s, g)) for b in BATCHES for s in TRAIN_SUBSETS
        for g in SEEDS]
rows = ["| run | batch | images | epochs | estimated duration |", "|---|---|---|---|---|"]
total = 0
for b, s, g, name in PLAN:
    n = min(s, len(samples)) if s else len(samples)
    sec = estimate(ITERS, b, DEVICE)
    total += sec or 0
    done = (Path(OUT) / "runs" / name / "final.weights").exists()
    rows.append(f"| {name}{' (done)' if done else ''} | {b} | {n} | "
                f"{ITERS * b / n:.2f} | "
                + (f"{sec / 3600:.1f} h |" if sec else "not measured |"))
display(Markdown("\n".join(rows)))
print(f"{len(PLAN)} runs of {ITERS} iterations"
      + (f", ≈ {total / 3600:.1f} h on {DEVICE} in total" if total else ""))

## Trainings

Each run goes to `OUT/runs/<run>/` (`b16-sall`, `b8-s500`…), with its `run.json`.
Running the cell again resumes the interrupted run (`--resume`) and skips the
finished ones (`SKIP_DONE`). On Colab with `DRIVE_DIR`, `OUT` is copied to
`<DRIVE_DIR>/runs/` every 10 min and at the end of the cell (T14.8).

In [ ]:
SYNC = COLAB and bool(DRIVE_DIR)
if SYNC:
    env.mount_drive(DRIVE_DIR)
    colab.restore_outputs(OUT, DRIVE_DIR)
stop = threading.Event()


def sync_loop(every=600):
    while not stop.wait(every):
        colab.sync_outputs(OUT, DRIVE_DIR)


if SYNC:
    threading.Thread(target=sync_loop, daemon=True).start()
try:
    for b, s, g, name in PLAN:
        run_dir = Path(OUT) / "runs" / name
        if SKIP_DONE and (run_dir / "final.weights").exists():
            print(f"{name}: already trained, skipped")
            continue
        resume = (run_dir / "checkpoint.npz").exists()
        runs.write_meta(run_dir, batch=b, subset=s, seed=g, iters=ITERS, lr=LR,
                        device=DEVICE, net=NET, dataset=DATASET, rev=TRACE["rev"])
        C.run(C.cmd_train(NET, run_dir, DATASET, INIT, INIT_NET, resume, iters=ITERS,
                          batch=b, lr=LR, burn_in=BURN_IN, multiscale=MULTISCALE,
                          subset=s, save_every=SAVE_EVERY, workers=WORKERS,
                          device=DEVICE, data_root=DATA_ROOT, seed=g),
              log=run_dir / "log.txt")
finally:
    stop.set()
    if SYNC:
        colab.sync_outputs(OUT, DRIVE_DIR)

## Comparison

Float mAP of each run on the same `SUBSET` images (`tools/eval_voc.py`, in
`OUT/eval/<run>/`, a cache shared with the comparison of the `_infer` notebooks:
reused if already computed), final loss and speed; loss curves overlaid
(`tools.figures.resultats.plot_training`, M13). The inference notebooks
`<model>_infer` read these runs (`RUN`, `COMPARE`).

In [ ]:
from tools.figures import MissingSource
from tools.figures import resultats as FR

RESULTS = [runs.evaluate(r, NET, DATASET, RESIZE, SUBSET, METRIC, SPLIT, SIZE, DATA_ROOT,
                         out_dir=f"{OUT}/eval/{r.name}", jobs=JOBS)
           for r in runs.find_runs(OUT) if r.name != runs.TRAIN]
display(Markdown(runs.table(RESULTS)))
curves = []
for b, s, g, name in PLAN:
    try:
        curves.append(FR.load_training(Path(OUT) / "runs" / name))
    except MissingSource:
        pass
if curves:
    for png in FR.plot_training(curves, Path(OUT) / "figures", "balayage"):
        if str(png).endswith(".png"):
            display(Image.open(png))

## Summary

In [ ]:
print(f"runs: {OUT}/runs/; backend {DEVICE}; revision {TRACE['rev']}")
best = max((r for r in RESULTS if r["mAP"] is not None), key=lambda r: r["mAP"],
           default=None)
if best:
    print(f"best run on {SUBSET or 'all'} images: {best['run']} "
          f"(mAP {best['mAP']:.2f}); inference: RUN = {best['run']!r}")
print("equivalent commands:")
for c in C.HISTORY:
    print("  " + c)